## Model Building and Evaluation

This notebook focuses on preparing the data, training multiple machine learning models, and selecting the best model for AQI prediction.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load data
data = pd.read_csv("/content/delhi-weather-aqi-2025.csv")
data.head()

,date_ist,time_ist,location,lat,lon,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,description,aqi_index,pm2_5,pm10,co,no2
0,01/01/2025,0:00,Anand Vihar,28.6469,77.316,8.1,100,995.4,2.9,Mainly clear,WMO Code 1,197,185.8,188.6,1907,56.7
1,01/01/2025,1:00,Anand Vihar,28.6469,77.316,7.7,100,994.7,3.2,Overcast,WMO Code 3,198,174.6,177.4,1669,44.8
2,01/01/2025,2:00,Anand Vihar,28.6469,77.316,7.5,100,994.3,4.5,Overcast,WMO Code 3,199,164.4,166.7,1493,34.6
3,01/01/2025,3:00,Anand Vihar,28.6469,77.316,7.8,99,994.1,6.0,Overcast,WMO Code 3,200,156.5,158.8,1401,26.7
4,01/01/2025,4:00,Anand Vihar,28.6469,77.316,7.3,100,993.8,6.8,Overcast,WMO Code 3,200,149.5,151.8,1372,20.6


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52560 entries, 0 to 52559
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date_ist        52560 non-null  object 
 1   time_ist        52560 non-null  object 
 2   location        52560 non-null  object 
 3   lat             52560 non-null  float64
 4   lon             52560 non-null  float64
 5   temp_c          52560 non-null  float64
 6   humidity        52560 non-null  int64  
 7   pressure_mb     52560 non-null  float64
 8   windspeed_kph   52560 non-null  float64
 9   condition_text  52560 non-null  object 
 10  description     52560 non-null  object 
 11  aqi_index       52560 non-null  int64  
 12  pm2_5           52560 non-null  float64
 13  pm10            52560 non-null  float64
 14  co              52560 non-null  int64  
 15  no2             52560 non-null  float64
dtypes: float64(8), int64(3), object(5)
memory usage: 6.4+ MB


## Data Preprocessing

In [ ]:
data.isnull().sum()

,0
date_ist,0
time_ist,0
location,0
lat,0
lon,0
temp_c,0
humidity,0
pressure_mb,0
windspeed_kph,0
condition_text,0


In [ ]:
# converting data types
data['hour'] = pd.to_datetime(data['time_ist'], format='%H:%M').dt.hour

In [ ]:
data['date_ist'] = pd.to_datetime(data['date_ist'], format='%d/%m/%Y')


In [ ]:
data['month'] = data['date_ist'].dt.month  # extracting month from date_ist

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52560 entries, 0 to 52559
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date_ist        52560 non-null  datetime64[ns]
 1   time_ist        52560 non-null  object        
 2   location        52560 non-null  object        
 3   lat             52560 non-null  float64       
 4   lon             52560 non-null  float64       
 5   temp_c          52560 non-null  float64       
 6   humidity        52560 non-null  int64         
 7   pressure_mb     52560 non-null  float64       
 8   windspeed_kph   52560 non-null  float64       
 9   condition_text  52560 non-null  object        
 10  description     52560 non-null  object        
 11  aqi_index       52560 non-null  int64         
 12  pm2_5           52560 non-null  float64       
 13  pm10            52560 non-null  float64       
 14  co              52560 non-null  int64         
 15  no

In [ ]:
data.drop(['time_ist'], axis=1, inplace=True)

In [ ]:
data.drop(['date_ist', 'description'], axis=1, inplace=True)

- The date_ist column was dropped after extracting month , making the original column redundant.
- The description column was removed because it contains the same information as condition_text.

In [ ]:
data.drop(['lat', 'lon'], axis=1, inplace=True)
## Latitude and longitude were dropped due to weak correlation with AQI,
##  limited impact on prediction.

In [ ]:
data.dtypes

,0
location,object
temp_c,float64
humidity,int64
pressure_mb,float64
windspeed_kph,float64
condition_text,object
aqi_index,int64
pm2_5,float64
pm10,float64
co,int64


In [ ]:
data.head()

,location,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,aqi_index,pm2_5,pm10,co,no2,hour,month
0,Anand Vihar,8.1,100,995.4,2.9,Mainly clear,197,185.8,188.6,1907,56.7,0,1
1,Anand Vihar,7.7,100,994.7,3.2,Overcast,198,174.6,177.4,1669,44.8,1,1
2,Anand Vihar,7.5,100,994.3,4.5,Overcast,199,164.4,166.7,1493,34.6,2,1
3,Anand Vihar,7.8,99,994.1,6.0,Overcast,200,156.5,158.8,1401,26.7,3,1
4,Anand Vihar,7.3,100,993.8,6.8,Overcast,200,149.5,151.8,1372,20.6,4,1


## Feature Encoding and Scaling

In [ ]:
# splitting data into X and y
X = data.drop('aqi_index', axis=1) # features
y = data['aqi_index'] # target

In [ ]:
# X = pd.get_dummies(data, columns=['location', 'condition_text'], drop_first=True)

In [ ]:
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

In [ ]:
one_hot_encoded = encoder.fit_transform(X[categorical_columns])

In [ ]:
# Convert to DataFrame
one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(categorical_columns)
)

In [ ]:
# Combine
X_encoded = pd.concat([X, one_hot_df], axis=1)

In [ ]:
# Drop original categorical columns
X_encoded = X_encoded.drop(categorical_columns, axis=1)

In [ ]:
X_encoded.dtypes

,0
temp_c,float64
humidity,int64
pressure_mb,float64
windspeed_kph,float64
pm2_5,float64
pm10,float64
co,int64
no2,float64
hour,int32
month,int32


In [ ]:
encoded_cols = one_hot_df.columns.tolist()

numerical_cols = [col for col in X_encoded.columns if col not in encoded_cols]

In [ ]:
## scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [ ]:
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])

In [ ]:
X_scaled = X_encoded.copy()

In [ ]:
X_scaled.head()

,temp_c,humidity,pressure_mb,windspeed_kph,pm2_5,pm10,co,no2,hour,month,...,condition_text_Drizzle: Dense,condition_text_Drizzle: Light,condition_text_Drizzle: Moderate,condition_text_Fog,condition_text_Mainly clear,condition_text_Overcast,condition_text_Partly cloudy,condition_text_Rain: Heavy,condition_text_Rain: Moderate,condition_text_Rain: Slight
0,-2.221588,1.607971,1.829608,-1.053992,1.738653,-0.270999,1.736164,0.733549,-1.661325,-1.602745,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,-2.274094,1.607971,1.724951,-0.967737,1.536353,-0.298132,1.340051,0.322902,-1.516862,-1.602745,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-2.300347,1.607971,1.665147,-0.593962,1.352116,-0.324054,1.047128,-0.029082,-1.372399,-1.602745,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-2.260967,1.565335,1.635245,-0.162684,1.209422,-0.343193,0.894009,-0.301697,-1.227936,-1.602745,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-2.326600,1.607971,1.590392,0.067331,1.082984,-0.360151,0.845743,-0.512197,-1.083473,-1.602745,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
X_scaled.dtypes

,0
temp_c,float64
humidity,float64
pressure_mb,float64
windspeed_kph,float64
pm2_5,float64
pm10,float64
co,float64
no2,float64
hour,float64
month,float64


## Train-Test Split


In [ ]:
## Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(42048, 25)
(10512, 25)


## Model Training


In [ ]:
## model--Linear regression
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

In [ ]:
lr_model.fit(X_train, y_train)

LinearRegression()

In [ ]:
y_pred = lr_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 96.84687809002975
MSE: 30963.80272372464
RMSE: 175.96534523514748
R2 Score: 0.7111674166403265


In [ ]:
## model-- Decision Tree regression
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(random_state=42)

In [ ]:
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [ ]:
y_pred_dt = dt_model.predict(X_test)

In [ ]:
mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = mse_dt ** 0.5
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree Results:")
print("MAE:", mae_dt)
print("RMSE:", rmse_dt)
print("R2 Score:", r2_dt)

Decision Tree Results:
MAE: 35.1988203957382
RMSE: 128.41611292731426
R2 Score: 0.8461735800221968


In [ ]:
print("Train Score:", dt_model.score(X_train, y_train))
print("Test Score:", dt_model.score(X_test, y_test))

Train Score: 1.0
Test Score: 0.8461735800221968


In [ ]:
## model-- Random Forest
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
y_pred_rf = rf_model.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mse_rf ** 0.5
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)

Random Forest Results:
MAE: 36.13630898021309
RMSE: 91.89053192478586
R2 Score: 0.9212349734226006


In [ ]:
print("Train Score:", rf_model.score(X_train, y_train))
print("Test Score:", rf_model.score(X_test, y_test))

Train Score: 0.9889525356936155
Test Score: 0.9212349734226006


In [ ]:
## model-- XGBoost
from xgboost import XGBRegressor
xgb_model = XGBRegressor(random_state=42)

In [ ]:
xgb_model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

In [ ]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = mse_xgb ** 0.5
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost Results:")
print("MAE:", mae_xgb)
print("RMSE:", rmse_xgb)
print("R2 Score:", r2_xgb)

XGBoost Results:
MAE: 47.84158706665039
RMSE: 95.46323633832817
R2 Score: 0.9149911403656006


In [ ]:
print("Train Score:", xgb_model.score(X_train, y_train))
print("Test Score:", xgb_model.score(X_test, y_test))

Train Score: 0.9624215364456177
Test Score: 0.9149911403656006


## Model Comparison


In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'R2 Score': [r2, r2_dt, r2_rf, r2_xgb],
    'RMSE': [rmse, rmse_dt, rmse_rf, rmse_xgb],
    'MAE': [mae, mae_dt, mae_rf, mae_xgb]
})

results

,Model,R2 Score,RMSE,MAE
0,Linear Regression,0.711167,175.965345,96.846878
1,Decision Tree,0.846174,128.416113,35.198820
2,Random Forest,0.921235,91.890532,36.136309
3,XGBoost,0.914991,95.463236,47.841587


- Random Forest Regressor achieved the best performance with the highest R² score (0.92) and lowest RMSE.

- After comparing multiple models, Random Forest Regressor was selected as the best-performing model. We then applied hyperparameter tuning to improve its performance.


## Hyperparameter Tuning


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

In [ ]:
param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

In [ ]:
rf = RandomForestRegressor(random_state=42)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

In [ ]:
grid.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 150]},
             scoring='r2')

In [ ]:
print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 150}


In [ ]:
best_rf = grid.best_estimator_

In [ ]:
y_pred_tuned = best_rf.predict(X_test) # predict with tuned model

In [ ]:
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
mse_tuned = mean_squared_error(y_test, y_pred_tuned)
rmse_tuned = mse_tuned ** 0.5
r2_tuned = r2_score(y_test, y_pred_tuned)

print("Tuned Random Forest Results:")
print("MAE:", mae_tuned)
print("RMSE:", rmse_tuned)
print("R2 Score:", r2_tuned)

Tuned Random Forest Results:
MAE: 35.979847158802635
RMSE: 91.39663502045302
R2 Score: 0.9220793968142432


After tuning the model using GridSearchCV, the performance improved slightly. The R² score increased and the prediction errors decreased, showing that tuning helped make the model better.

## Final Model Pipeline

Pipeline is used to combine preprocessing (scaling and encoding) with the trained model, making prediction easier and more consistent.


In [ ]:
df = data.copy()

In [ ]:
X = df.drop("aqi_index", axis=1)
y = df["aqi_index"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    random_state=42
)
# {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 150}

In [ ]:
numerical_cols = [
    "temp_c", "humidity", "pressure_mb", "windspeed_kph",
    "pm2_5", "pm10", "co", "no2", "hour", "month"
]

categorical_cols = [
    "location", "condition_text"
]

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf)
])

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['temp_c', 'humidity',
                                                   'pressure_mb',
                                                   'windspeed_kph', 'pm2_5',
                                                   'pm10', 'co', 'no2', 'hour',
                                                   'month']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['location',
                                                   'condition_text'])])),
                ('model',
                 RandomForestRegressor(n_estimators=150, random_state=42))])

In [ ]:
import joblib

joblib.dump(pipeline, "aqi_pipeline.pkl")

['aqi_pipeline.pkl']

In [ ]:
pipeline = joblib.load("aqi_pipeline.pkl")
# load and predict

In [ ]:
input_data = pd.DataFrame([{
    "location": "Dwarka",
    "condition_text": "Fog",
    "temp_c": 32,
    "humidity": 70,
    "pressure_mb": 1012,
    "windspeed_kph": 10,
    "pm2_5": 120,
    "pm10": 100,
    "co": 1,
    "no2": 40,
    "hour": 14,
    "month": 1
}])

In [ ]:
prediction = pipeline.predict(input_data)
print("Predicted AQI:", prediction[0])

Predicted AQI: 217.44666666666666


In [ ]:
sample = pd.DataFrame([{
    "location": "Dwarka",
    "condition_text": "Fog",
    "temp_c": 32,
    "humidity": 70,
    "pressure_mb": 1012,
    "windspeed_kph": 10,
    "pm2_5": 120,
    "pm10": 180,
    "co": 1,
    "no2": 40,
    "hour": 14,
    "month": 4
}])

In [ ]:
prediction = pipeline.predict(sample)
print(prediction)

[365.92666667]


## Conclusion:

Random Forest performed best, and a pipeline was used to make prediction simple and consistent.
